# 01. 데이터 탐색 (Exploratory Data Analysis)

BC카드 전국 시군구별·업종별 소비 집계데이터(`ABP_CONTEST_DATA.csv`)를 로드하고,
공식 안내문서와 실측 데이터 사이의 불일치(§2-1 실측 함정)를 직접 확인한다.


In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from data_loader import load_raw_data, validate_business_categories

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)


## 1-1. 원본 데이터 로딩 및 스키마 검증

`data_loader.load_raw_data()`는 로딩과 동시에 컬럼명·값 범위를 assert로 검증한다.

In [2]:
raw = load_raw_data()
print(f"행 수: {len(raw):,}")
print(f"컬럼: {raw.columns.tolist()}")
raw.head()


행 수: 242,574
컬럼: ['STRD_YYMM', 'SIDO_NM', 'CCG_NM', 'GENDER_CD', 'AGE_CD', 'TP_BUZ_NO', 'TP_BUZ_NM', 'amt', 'cnt']


,STRD_YYMM,SIDO_NM,CCG_NM,GENDER_CD,AGE_CD,TP_BUZ_NO,TP_BUZ_NM,amt,cnt
0,202605,서울특별시,용산구,2,3,8006,서양음식,734430000,38874
1,202605,부산광역시,사상구,2,4,8001,일반한식,424220000,10929
2,202601,경상북도,포항시 북구,1,5,4010,편 의 점,283030000,29848
3,202601,전라남도,여수시,1,6,4010,편 의 점,204060000,20727
4,202606,서울특별시,송파구,1,3,4010,편 의 점,575120000,86780


## 1-2. 실측 함정 #1: 컬럼명 대소문자

공식 안내문서(`ABP_CONTEST_DATA 설명`)는 `AMT`, `CNT` 대문자로 안내하지만,
실제 파일의 금액/건수 컬럼명은 **소문자** `amt`, `cnt` 이다.

In [3]:
assert 'amt' in raw.columns and 'cnt' in raw.columns
assert 'AMT' not in raw.columns and 'CNT' not in raw.columns
print("[확인] 실제 컬럼명은 소문자 amt/cnt 이다 (공식 안내문서와 불일치).")


[확인] 실제 컬럼명은 소문자 amt/cnt 이다 (공식 안내문서와 불일치).


## 1-3. 실측 함정 #2: 외국인(GENDER_CD='3')의 AGE_CD

공식 안내문서는 "외국인은 AGE_CD = x"라고 안내하지만, 실제로는 외국인도 연령대(1~6)가
정상적으로 채워져 있다. 이는 **외국인 연령대별 분석이 가능하다는 중요한 기회**다.

In [4]:
foreign_age = raw.loc[raw['GENDER_CD'] == '3', 'AGE_CD'].value_counts().sort_index()
corp_age = raw.loc[raw['GENDER_CD'] == 'x', 'AGE_CD'].value_counts()

print("외국인(GENDER_CD=3)의 AGE_CD 분포:")
print(foreign_age)
print("\n법인(GENDER_CD=x)의 AGE_CD 분포:")
print(corp_age)
print("\n[확인] 'x'는 법인에서만 나타나고, 외국인은 1~6이 모두 채워져 있다 (공식 안내문서와 불일치).")


외국인(GENDER_CD=3)의 AGE_CD 분포:
AGE_CD
1     7566
2    12152
3    12564
4    12641
5    12702
6    12553
Name: count, dtype: int64

법인(GENDER_CD=x)의 AGE_CD 분포:
AGE_CD
x    13161
Name: count, dtype: int64

[확인] 'x'는 법인에서만 나타나고, 외국인은 1~6이 모두 채워져 있다 (공식 안내문서와 불일치).


## 1-4. 실측 함정 #3: 업종명 공백 불규칙

11개 업종 중 일부는 원본에 불규칙한 공백이 섞여 있어(`"편 의 점"` 등) 그룹핑 전 정규화가 필요하다.

In [5]:
raw_unique = raw['TP_BUZ_NM'].unique()
print(f"정규화 전 고유 업종명 개수: {len(raw_unique)}개")
print(sorted(raw_unique))

normalized = raw['TP_BUZ_NM'].str.replace(' ', '', regex=False)
print(f"\n정규화 후 고유 업종명 개수: {normalized.nunique()}개")
print(sorted(normalized.unique()))


정규화 전 고유 업종명 개수: 11개
['갈비전문점', '대형할인점', '서양음식', '슈퍼 마켓', '스넥', '일반한식', '일식회집', '제 과 점', '중국음식', '편 의 점', '한정식']

정규화 후 고유 업종명 개수: 11개
['갈비전문점', '대형할인점', '서양음식', '슈퍼마켓', '스넥', '일반한식', '일식회집', '제과점', '중국음식', '편의점', '한정식']


## 1-5. ⚠️ 추가로 발견한 함정: 시군구명 중복 (SIDO_NM 없이 CCG_NM만으로 groupby 하면 안 됨)

"중구"·"동구"·"서구"·"남구"·"북구"·"강서구"·"고성군" 등 7개 시군구명은
서로 다른 광역시도에 동시에 존재한다. 이후 모든 지역 단위 분석은 반드시
`(SIDO_NM, CCG_NM)` 복합키를 사용해야 한다 (`feature_engineering.py`에 반영됨).

In [6]:
dup = raw.groupby('CCG_NM')['SIDO_NM'].nunique()
dup_names = dup[dup > 1]
print(f"여러 시도에 걸쳐 중복되는 시군구명: {len(dup_names)}개")
print(dup_names)


여러 시도에 걸쳐 중복되는 시군구명: 7개
CCG_NM
강서구    2
고성군    2
남구     4
동구     6
북구     4
서구     5
중구     6
Name: SIDO_NM, dtype: int64


## 1-6. 값 범위 및 결측 확인

In [7]:
print("기준년월:", sorted(raw['STRD_YYMM'].unique()))
print("\n성별 분포:\n", raw['GENDER_CD'].value_counts())
print("\n결측치:\n", raw.isnull().sum())
print("\namt<=0 개수:", (raw['amt'] <= 0).sum(), " / cnt<=0 개수:", (raw['cnt'] <= 0).sum())
print("\n시도 수:", raw['SIDO_NM'].nunique(), " / (시도,시군구) 조합 수:", raw.groupby(['SIDO_NM','CCG_NM']).ngroups)


기준년월: ['202601', '202602', '202603', '202604', '202605', '202606']

성별 분포:
 GENDER_CD
1    80554
2    78681
3    70178
x    13161
Name: count, dtype: int64



결측치:
 STRD_YYMM    0
SIDO_NM      0
CCG_NM       0
GENDER_CD    0
AGE_CD       0
TP_BUZ_NO    0
TP_BUZ_NM    0
amt          0
cnt          0
dtype: int64

amt<=0 개수: 0  / cnt<=0 개수: 0



시도 수: 17  / (시도,시군구) 조합 수: 255


## 1-7. 업종 11종 검증

In [8]:
validate_business_categories(raw)
print("[확인] 공백 정규화 후 업종은 정확히 11종으로 수렴한다.")


[확인] 공백 정규화 후 업종은 정확히 11종으로 수렴한다.


## 요약

- 실측 데이터는 공식 안내문서와 3가지 불일치가 있다 (컬럼명 대소문자, 외국인 AGE_CD, 업종명 공백).
- 시군구명은 `SIDO_NM` 없이 단독으로 쓰면 7개 지역에서 서로 다른 지역이 뒤섞인다.
- 결측치·0 이하 값은 없다 (전처리 단계에서 별도 대체 로직 불필요).

다음 단계(`02_preprocessing.ipynb`)에서 이 검증 결과를 바탕으로 외국인 소비 데이터를 정제한다.